In [51]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results"

EXPECTED_DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]

print("ROOT:", ROOT)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

ROOT: /home/stephane/resource-efficiency-benchmark
RESULTS_DIR exists: True


In [52]:
def discover_runs(results_dir: Path) -> pd.DataFrame:
    rows = []

    if not results_dir.exists():
        return pd.DataFrame(columns=["model", "run_id", "run_dir"])

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        model = model_dir.name

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            run_id = run_dir.name

            rows.append({
                "model": model,
                "run_id": run_id,
                "run_dir": str(run_dir),
            })

    return pd.DataFrame(rows)


def expected_files_for_run(run_dir: Path) -> dict:
    file_map = {
        "config_yaml": run_dir / "config.yaml",
        "server_log": run_dir / "server.log",
    }

    for ds in EXPECTED_DATASETS:
        file_map[f"{ds}_summary"] = run_dir / f"{ds}.csv"
        file_map[f"{ds}_samples"] = run_dir / f"{ds}_samples.csv"

    return file_map


def audit_run_files(runs_df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in runs_df.iterrows():
        run_dir = Path(row["run_dir"])
        file_map = expected_files_for_run(run_dir)

        record = {
            "model": row["model"],
            "run_id": row["run_id"],
            "run_dir": row["run_dir"],
        }

        for key, path in file_map.items():
            record[key] = path.exists()

        records.append(record)

    return pd.DataFrame(records)


def list_present_dataset_files(run_dir: Path):
    summary_files = []
    sample_files = []

    for ds in EXPECTED_DATASETS:
        s = run_dir / f"{ds}.csv"
        sp = run_dir / f"{ds}_samples.csv"

        if s.exists():
            summary_files.append((ds, s))
        if sp.exists():
            sample_files.append((ds, sp))

    return summary_files, sample_files


def safe_read_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"FAILED reading {path}: {e}")
        return pd.DataFrame()


def collect_file_inventory(runs_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in runs_df.iterrows():
        run_dir = Path(row["run_dir"])
        summary_files, sample_files = list_present_dataset_files(run_dir)

        for dataset, path in summary_files:
            rows.append({
                "model": row["model"],
                "run_id": row["run_id"],
                "dataset": dataset,
                "file_type": "summary",
                "path": str(path),
            })

        for dataset, path in sample_files:
            rows.append({
                "model": row["model"],
                "run_id": row["run_id"],
                "dataset": dataset,
                "file_type": "samples",
                "path": str(path),
            })

    return pd.DataFrame(rows)


def schema_inventory(files_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in files_df.iterrows():
        path = Path(row["path"])
        df = safe_read_csv(path)

        rows.append({
            "model": row["model"],
            "run_id": row["run_id"],
            "dataset": row["dataset"],
            "file_type": row["file_type"],
            "path": row["path"],
            "n_rows": len(df),
            "n_cols": len(df.columns),
            "columns": list(df.columns),
        })

    return pd.DataFrame(rows)


def explode_columns(schema_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in schema_df.iterrows():
        for col in row["columns"]:
            rows.append({
                "model": row["model"],
                "run_id": row["run_id"],
                "dataset": row["dataset"],
                "file_type": row["file_type"],
                "column": col,
            })

    return pd.DataFrame(rows)

In [53]:
runs_df = discover_runs(RESULTS_DIR)
runs_df

,model,run_id,run_dir
0,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,/home/stephane/resource-efficiency-benchmark/r...
1,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,/home/stephane/resource-efficiency-benchmark/r...
2,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,/home/stephane/resource-efficiency-benchmark/r...
3,Phi-4-mini-instruct-Q4_K_M,2026-04-02_00-18-49,/home/stephane/resource-efficiency-benchmark/r...
4,Phi-4-mini-instruct-Q4_K_M,2026-04-02_01-57-36,/home/stephane/resource-efficiency-benchmark/r...
5,Phi-4-mini-instruct-Q4_K_M,2026-04-02_02-45-58,/home/stephane/resource-efficiency-benchmark/r...
6,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,/home/stephane/resource-efficiency-benchmark/r...
7,gemma-3-1b-it-Q8_0,2026-03-31_15-37-23,/home/stephane/resource-efficiency-benchmark/r...
8,gemma-3-1b-it-Q8_0,2026-03-31_16-39-31,/home/stephane/resource-efficiency-benchmark/r...
9,gemma-3-1b-it-Q8_0,2026-04-02_02-45-58,/home/stephane/resource-efficiency-benchmark/r...


In [54]:
audit_df = audit_run_files(runs_df)
audit_df

,model,run_id,run_dir,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,True,True
1,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,/home/stephane/resource-efficiency-benchmark/r...,True,True,True,True,True,True,True,True,False,False
2,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,/home/stephane/resource-efficiency-benchmark/r...,True,True,True,True,True,True,False,False,False,False
3,Phi-4-mini-instruct-Q4_K_M,2026-04-02_00-18-49,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,True,True,False,False
4,Phi-4-mini-instruct-Q4_K_M,2026-04-02_01-57-36,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,False,False
5,Phi-4-mini-instruct-Q4_K_M,2026-04-02_02-45-58,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,True,True
6,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,/home/stephane/resource-efficiency-benchmark/r...,True,True,True,True,True,True,True,True,False,False
7,gemma-3-1b-it-Q8_0,2026-03-31_15-37-23,/home/stephane/resource-efficiency-benchmark/r...,True,True,True,True,True,True,True,True,False,False
8,gemma-3-1b-it-Q8_0,2026-03-31_16-39-31,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,False,False
9,gemma-3-1b-it-Q8_0,2026-04-02_02-45-58,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,True,True


In [55]:
if not audit_df.empty:
    display(
        audit_df.groupby("model").agg({
            "run_id": "nunique",
            "config_yaml": "sum",
            "server_log": "sum",
            "mmlu_summary": "sum",
            "mmlu_samples": "sum",
            "hellaswag_summary": "sum",
            "hellaswag_samples": "sum",
            "gsm8k_summary": "sum",
            "gsm8k_samples": "sum",
            "dolly_summary": "sum",
            "dolly_samples": "sum",
        }).rename(columns={"run_id": "n_runs"})
    )

,n_runs,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
model,,,,,,,,,,,
Llama-3.2-1B-Instruct-Q8_0,2,2,2,1,1,1,1,1,1,1,1
Phi-4-mini-instruct-Q4_K_M,4,4,4,1,1,1,1,1,1,1,1
gemma-3-1b-it-Q8_0,4,4,4,2,2,2,2,2,2,1,1
raw,9,0,0,0,0,0,0,0,0,0,0


In [56]:
required_common = ["config_yaml", "server_log"]
dataset_file_cols = [
    "mmlu_summary", "mmlu_samples",
    "hellaswag_summary", "hellaswag_samples",
    "gsm8k_summary", "gsm8k_samples",
    "dolly_summary", "dolly_samples",
]

incomplete_df = audit_df[
    ~(audit_df[required_common].all(axis=1))
    | ~(audit_df[dataset_file_cols].any(axis=1))
]

incomplete_df

,model,run_id,run_dir,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
4,Phi-4-mini-instruct-Q4_K_M,2026-04-02_01-57-36,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,False,False
8,gemma-3-1b-it-Q8_0,2026-03-31_16-39-31,/home/stephane/resource-efficiency-benchmark/r...,True,True,False,False,False,False,False,False,False,False
10,raw,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
11,raw,granite4_htiny_awq,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
12,raw,llama3_2_1b,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
13,raw,ministral3_3b_awq,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
14,raw,ministral3_8b_awq,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
15,raw,phi4mini_gptq_4bit,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
16,raw,qwen2_5_1_5b,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
17,raw,qwen2_7b_q4,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False


In [57]:
files_df = collect_file_inventory(runs_df)
files_df.head(20)

,model,run_id,dataset,file_type,path
0,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,summary,/home/stephane/resource-efficiency-benchmark/r...
1,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,samples,/home/stephane/resource-efficiency-benchmark/r...
2,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,summary,/home/stephane/resource-efficiency-benchmark/r...
3,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,summary,/home/stephane/resource-efficiency-benchmark/r...
4,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,summary,/home/stephane/resource-efficiency-benchmark/r...
5,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,samples,/home/stephane/resource-efficiency-benchmark/r...
6,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,samples,/home/stephane/resource-efficiency-benchmark/r...
7,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,samples,/home/stephane/resource-efficiency-benchmark/r...
8,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,mmlu,summary,/home/stephane/resource-efficiency-benchmark/r...
9,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,hellaswag,summary,/home/stephane/resource-efficiency-benchmark/r...


In [58]:
schema_df = schema_inventory(files_df)
schema_df[["model", "run_id", "dataset", "file_type", "n_rows", "n_cols", "path"]]

,model,run_id,dataset,file_type,n_rows,n_cols,path
0,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,summary,1,25,/home/stephane/resource-efficiency-benchmark/r...
1,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,samples,100,23,/home/stephane/resource-efficiency-benchmark/r...
2,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,summary,1,21,/home/stephane/resource-efficiency-benchmark/r...
3,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,summary,1,21,/home/stephane/resource-efficiency-benchmark/r...
4,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,summary,1,21,/home/stephane/resource-efficiency-benchmark/r...
5,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,samples,1000,23,/home/stephane/resource-efficiency-benchmark/r...
6,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,samples,1000,23,/home/stephane/resource-efficiency-benchmark/r...
7,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,samples,1000,23,/home/stephane/resource-efficiency-benchmark/r...
8,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,mmlu,summary,1,21,/home/stephane/resource-efficiency-benchmark/r...
9,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,hellaswag,summary,1,21,/home/stephane/resource-efficiency-benchmark/r...


In [59]:
summary_rows_check = schema_df[schema_df["file_type"] == "summary"][
    ["model", "run_id", "dataset", "n_rows", "path"]
].sort_values(["model", "run_id", "dataset"])

summary_rows_check

,model,run_id,dataset,n_rows,path
0,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,1,/home/stephane/resource-efficiency-benchmark/r...
4,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,1,/home/stephane/resource-efficiency-benchmark/r...
3,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,1,/home/stephane/resource-efficiency-benchmark/r...
2,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,1,/home/stephane/resource-efficiency-benchmark/r...
9,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,hellaswag,1,/home/stephane/resource-efficiency-benchmark/r...
8,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,mmlu,1,/home/stephane/resource-efficiency-benchmark/r...
12,Phi-4-mini-instruct-Q4_K_M,2026-04-02_00-18-49,gsm8k,1,/home/stephane/resource-efficiency-benchmark/r...
14,Phi-4-mini-instruct-Q4_K_M,2026-04-02_02-45-58,dolly,1,/home/stephane/resource-efficiency-benchmark/r...
18,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,gsm8k,1,/home/stephane/resource-efficiency-benchmark/r...
17,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,hellaswag,1,/home/stephane/resource-efficiency-benchmark/r...


In [60]:
bad_summary_rows = summary_rows_check[summary_rows_check["n_rows"] != 1]
bad_summary_rows

,model,run_id,dataset,n_rows,path


In [61]:
sample_rows_check = schema_df[schema_df["file_type"] == "samples"][
    ["model", "run_id", "dataset", "n_rows", "path"]
].sort_values(["model", "run_id", "dataset"])

sample_rows_check

,model,run_id,dataset,n_rows,path
1,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,100,/home/stephane/resource-efficiency-benchmark/r...
7,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,1000,/home/stephane/resource-efficiency-benchmark/r...
6,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,1000,/home/stephane/resource-efficiency-benchmark/r...
5,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,1000,/home/stephane/resource-efficiency-benchmark/r...
11,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,hellaswag,1000,/home/stephane/resource-efficiency-benchmark/r...
10,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,mmlu,1000,/home/stephane/resource-efficiency-benchmark/r...
13,Phi-4-mini-instruct-Q4_K_M,2026-04-02_00-18-49,gsm8k,1000,/home/stephane/resource-efficiency-benchmark/r...
15,Phi-4-mini-instruct-Q4_K_M,2026-04-02_02-45-58,dolly,100,/home/stephane/resource-efficiency-benchmark/r...
21,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,gsm8k,1000,/home/stephane/resource-efficiency-benchmark/r...
20,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,hellaswag,1000,/home/stephane/resource-efficiency-benchmark/r...


In [62]:
columns_df = explode_columns(schema_df)

columns_by_group = (
    columns_df.groupby(["dataset", "file_type"])["column"]
    .unique()
    .reset_index()
)

columns_by_group

,dataset,file_type,column
0,dolly,samples,"[model_name, dataset_name, run_idx, sample_idx..."
1,dolly,summary,"[Model name, Dolly TTFT ms mean, Dolly TTFT ms..."
2,gsm8k,samples,"[model_name, dataset_name, run_idx, sample_idx..."
3,gsm8k,summary,"[Model name, Accuracy mean, Accuracy sd, TTFT ..."
4,hellaswag,samples,"[model_name, dataset_name, run_idx, sample_idx..."
5,hellaswag,summary,"[Model name, Accuracy mean, Accuracy sd, TTFT ..."
6,mmlu,samples,"[model_name, dataset_name, run_idx, sample_idx..."
7,mmlu,summary,"[Model name, Accuracy mean, Accuracy sd, TTFT ..."


In [63]:
schema_signature_df = schema_df.copy()
schema_signature_df["schema_signature"] = schema_signature_df["columns"].apply(lambda cols: tuple(cols))

schema_consistency = (
    schema_signature_df.groupby(["dataset", "file_type"])["schema_signature"]
    .nunique()
    .reset_index(name="n_distinct_schemas")
)

schema_consistency

,dataset,file_type,n_distinct_schemas
0,dolly,samples,1
1,dolly,summary,1
2,gsm8k,samples,1
3,gsm8k,summary,1
4,hellaswag,samples,1
5,hellaswag,summary,1
6,mmlu,samples,1
7,mmlu,summary,1


inconsistent_schemas = schema_consistency[schema_consistency["n_distinct_schemas"] > 1]
inconsistent_schemas

In [64]:
for dataset in EXPECTED_DATASETS:
    for file_type in ["summary", "samples"]:
        subset = files_df[(files_df["dataset"] == dataset) & (files_df["file_type"] == file_type)]
        if subset.empty:
            continue

        path = Path(subset.iloc[0]["path"])
        df = safe_read_csv(path)

        print(f"\n=== {dataset} / {file_type} ===")
        print(path)
        display(df.head(3))


=== mmlu / summary ===
/home/stephane/resource-efficiency-benchmark/results/Llama-3.2-1B-Instruct-Q8_0/2026-04-02_04-21-16/mmlu.csv


,Model name,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak
0,Llama-3.2-1B-Instruct-Q8_0.gguf,0.411,0.0099,150.3767,28.7989,269.0623,28.2029,8.5329,0.0243,67.0688,0.6122,62.3364,0.177,3.7334,0.3687,3.7065,0.3547,100.0,3316.52,182.91,64.8



=== mmlu / samples ===
/home/stephane/resource-efficiency-benchmark/results/Llama-3.2-1B-Instruct-Q8_0/2026-04-02_04-21-16/mmlu_samples.csv


,model_name,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak
0,Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2,C,True,C,1,125.7032,240.7181,8.6945,64.9667,100.0,3244.31,180.36,58.9000,59.8
1,Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2,C,True,C,1,124.5592,241.3399,8.5631,63.9667,97.9,3245.33,180.36,60.7333,60.9
2,Llama-3.2-1B-Instruct-Q8_0.gguf,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1,B,True,B,1,133.7754,241.4040,9.2912,96.8667,100.0,3248.53,180.36,60.7333,61.5



=== hellaswag / summary ===
/home/stephane/resource-efficiency-benchmark/results/Llama-3.2-1B-Instruct-Q8_0/2026-04-02_04-21-16/hellaswag.csv


...

,model_name,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak
0,Llama-3.2-1B-Instruct-Q8_0.gguf,gsm8k,1,1,Donny can only drink water if it's at least 40...,Donny can only drink water if it's at least 40...,NaN,NaN,NaN,68,40.0,False,40,1,130.5718,243.2559,8.8744,64.1333,97.8,4187.59,186.48,60.2000,61.5
1,Llama-3.2-1B-Instruct-Q8_0.gguf,gsm8k,1,2,There are three trees in Eddy's backyard. The ...,There are three trees in Eddy's backyard. The ...,NaN,NaN,NaN,34,18.0,False,18,1,128.7144,244.3457,8.6482,63.9667,97.8,4189.91,186.48,62.0333,62.6
2,Llama-3.2-1B-Instruct-Q8_0.gguf,gsm8k,1,3,Cars have lined up on the motorway. Some of th...,Cars have lined up on the motorway. Some of th...,NaN,NaN,NaN,5,20.0,False,20,1,142.5976,259.7667,8.5347,62.6333,100.0,4194.39,186.48,62.6000,62.6



=== dolly / summary ===
/home/stephane/resource-efficiency-benchmark/results/Llama-3.2-1B-Instruct-Q8_0/2026-04-02_02-45-58/dolly.csv


,Model name,Dolly TTFT ms mean,Dolly TTFT ms sd,Dolly End-to-end latency ms mean,Dolly End-to-end latency ms sd,Dolly Tokens/sec mean,Dolly Tokens/sec sd,Dolly CPU percent avg mean,Dolly CPU percent avg sd,Dolly Temp C avg mean,Dolly Temp C avg sd,Dolly Sequential sustained tokens/sec mean,Dolly Sequential sustained tokens/sec sd,Dolly Sequential sustained samples/sec mean,Dolly Sequential sustained samples/sec sd,Dolly CPU percent peak,Dolly RAM used MB peak,Dolly Process RAM MB peak,Dolly Temp C peak,llama_bench model,llama_bench n_prompt,llama_bench n_gen,llama_bench prompt_tps,llama_bench gen_tps,llama_bench test_time_s
0,Llama-3.2-1B-Instruct-Q8_0.gguf,2059.4789,1611.0251,8217.0189,2087.5888,6.0186,0.3926,92.1132,1.8935,64.183,0.9265,4.616,0.8917,0.1298,0.0378,100.0,2514.2,193.52,68.6,models/Llama-3.2-1B-Instruct-Q8_0.gguf,0,128,NaN,NaN,2026-04-02T01:02:28Z



=== dolly / samples ===
/home/stephane/resource-efficiency-benchmark/results/Llama-3.2-1B-Instruct-Q8_0/2026-04-02_02-45-58/dolly_samples.csv


,model_name,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak
0,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,1,Identify which instrument is string or percuss...,NaN,NaN,Identify which instrument is string or percuss...,NaN,NaN,NaN,NaN,Tifa is a string instrument.,5,388.8920,1193.7671,6.2121,82.800,100.0,2033.75,183.55,58.4500,59.8
1,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,2,Give me a list of the Mercury Prize winning al...,NaN,NaN,Give me a list of the Mercury Prize winning al...,NaN,NaN,NaN,NaN,Here is the list of Mercury Prize winners sinc...,39,484.1469,9066.5494,4.5442,94.245,100.0,2033.75,183.56,60.8250,61.5
2,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,3,What should I do on a free afternoon in San Fr...,NaN,NaN,What should I do on a free afternoon in San Fr...,NaN,NaN,NaN,NaN,Here are some ideas for things to do on a free...,56,390.4580,8954.7648,6.5388,95.530,100.0,2029.58,183.58,61.4875,62.6


In [65]:
summary_tables = []

for _, row in files_df[files_df["file_type"] == "summary"].iterrows():
    path = Path(row["path"])
    df = safe_read_csv(path)

    if df.empty:
        continue

    df["model_dir"] = row["model"]
    df["run_id"] = row["run_id"]
    df["dataset"] = row["dataset"]
    df["source_path"] = str(path)

    summary_tables.append(df)

all_summaries_df = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
all_summaries_df.head()

,Model name,Dolly TTFT ms mean,Dolly TTFT ms sd,Dolly End-to-end latency ms mean,Dolly End-to-end latency ms sd,Dolly Tokens/sec mean,Dolly Tokens/sec sd,Dolly CPU percent avg mean,Dolly CPU percent avg sd,Dolly Temp C avg mean,Dolly Temp C avg sd,Dolly Sequential sustained tokens/sec mean,Dolly Sequential sustained tokens/sec sd,Dolly Sequential sustained samples/sec mean,Dolly Sequential sustained samples/sec sd,Dolly CPU percent peak,Dolly RAM used MB peak,Dolly Process RAM MB peak,Dolly Temp C peak,llama_bench model,llama_bench n_prompt,llama_bench n_gen,llama_bench prompt_tps,llama_bench gen_tps,llama_bench test_time_s,model_dir,run_id,dataset,source_path,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak
0,Llama-3.2-1B-Instruct-Q8_0.gguf,2059.4789,1611.0251,8217.0189,2087.5888,6.0186,0.3926,92.1132,1.8935,64.183,0.9265,4.616,0.8917,0.1298,0.0378,100.0,2514.2,193.52,68.6,models/Llama-3.2-1B-Instruct-Q8_0.gguf,0.0,128.0,NaN,NaN,2026-04-02T01:02:28Z,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-1B-Instruct-Q8_0.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,mmlu,/home/stephane/resource-efficiency-benchmark/r...,0.411,0.0099,150.3767,28.7989,269.0623,28.2029,8.5329,0.0243,67.0688,0.6122,62.3364,0.1770,3.7334,0.3687,3.7065,0.3547,100.0,3316.52,182.91,64.8
2,Llama-3.2-1B-Instruct-Q8_0.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,hellaswag,/home/stephane/resource-efficiency-benchmark/r...,0.325,0.0085,131.0324,0.8723,248.7212,1.3203,8.5040,0.0450,66.9892,0.6202,62.2864,0.2558,3.9748,0.0213,3.9748,0.0213,100.0,3948.59,185.95,64.8
3,Llama-3.2-1B-Instruct-Q8_0.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_04-21-16,gsm8k,/home/stephane/resource-efficiency-benchmark/r...,0.060,0.0000,127.1764,0.6863,298.7926,4.4268,7.2907,0.0482,68.6174,0.6857,63.7861,0.2294,3.3925,0.0198,3.3200,0.0507,100.0,4227.94,188.27,65.9
4,Phi-4-mini-instruct-Q4_K_M.gguf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Phi-4-mini-instruct-Q4_K_M,2026-03-31_12-29-17,mmlu,/home/stephane/resource-efficiency-benchmark/r...,0.599,0.0074,293.6792,80.1414,1127.2051,80.3621,3.5550,0.0113,73.7375,0.5077,66.5634,0.4633,2.6288,0.1763,0.8890,0.0599,100.0,6154.28,145.00,70.3


In [66]:
sample_tables = []

for _, row in files_df[files_df["file_type"] == "samples"].iterrows():
    path = Path(row["path"])
    df = safe_read_csv(path)

    if df.empty:
        continue

    df["model_dir"] = row["model"]
    df["run_id"] = row["run_id"]
    df["dataset"] = row["dataset"]
    df["source_path"] = str(path)

    sample_tables.append(df)

all_samples_df = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()
all_samples_df.head()

,model_name,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,model_dir,run_id,dataset,source_path
0,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,1,Identify which instrument is string or percuss...,NaN,NaN,Identify which instrument is string or percuss...,NaN,NaN,NaN,NaN,Tifa is a string instrument.,5,388.8920,1193.7671,6.2121,82.8000,100.0,2033.75,183.55,58.4500,59.8,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,/home/stephane/resource-efficiency-benchmark/r...
1,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,2,Give me a list of the Mercury Prize winning al...,NaN,NaN,Give me a list of the Mercury Prize winning al...,NaN,NaN,NaN,NaN,Here is the list of Mercury Prize winners sinc...,39,484.1469,9066.5494,4.5442,94.2450,100.0,2033.75,183.56,60.8250,61.5,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,/home/stephane/resource-efficiency-benchmark/r...
2,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,3,What should I do on a free afternoon in San Fr...,NaN,NaN,What should I do on a free afternoon in San Fr...,NaN,NaN,NaN,NaN,Here are some ideas for things to do on a free...,56,390.4580,8954.7648,6.5388,95.5300,100.0,2029.58,183.58,61.4875,62.6,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,/home/stephane/resource-efficiency-benchmark/r...
3,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,4,Give me a list of actresses who won an Oscar i...,NaN,NaN,Give me a list of actresses who won an Oscar i...,NaN,NaN,NaN,NaN,Here is the list of actresses who won an Oscar...,42,433.5472,9018.1888,4.8925,94.2425,100.0,2034.08,183.59,61.8875,62.6,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,/home/stephane/resource-efficiency-benchmark/r...
4,Llama-3.2-1B-Instruct-Q8_0.gguf,dolly,1,5,Give me a list of the ways you can be 'out' in...,NaN,NaN,Give me a list of the ways you can be 'out' in...,NaN,NaN,NaN,NaN,"Here's a list of ways you can be ""out"" in cric...",51,382.5714,8973.7716,5.9363,94.2125,100.0,2033.33,183.59,62.0350,63.1,Llama-3.2-1B-Instruct-Q8_0,2026-04-02_02-45-58,dolly,/home/stephane/resource-efficiency-benchmark/r...


In [67]:
if not all_summaries_df.empty:
    print("Summary shape:", all_summaries_df.shape)
    print("\nDatasets in summaries:", sorted(all_summaries_df["dataset"].dropna().unique()))
    print("\nRuns per model:")
    display(all_summaries_df.groupby("model_dir")["run_id"].nunique().rename("n_runs").reset_index())

    print("\nNull counts:")
    display(all_summaries_df.isna().sum().sort_values(ascending=False).head(30))

Summary shape: (15, 49)

Datasets in summaries: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']

Runs per model:


,model_dir,n_runs
0,Llama-3.2-1B-Instruct-Q8_0,2
1,Phi-4-mini-instruct-Q4_K_M,3
2,gemma-3-1b-it-Q8_0,3



Null counts:


llama_bench gen_tps                            15
llama_bench prompt_tps                         15
Dolly TTFT ms sd                               12
Dolly Temp C avg mean                          12
Dolly End-to-end latency ms sd                 12
Dolly Tokens/sec mean                          12
Dolly Tokens/sec sd                            12
Dolly End-to-end latency ms mean               12
Dolly CPU percent avg sd                       12
Dolly Sequential sustained samples/sec mean    12
Dolly Temp C avg sd                            12
Dolly Sequential sustained tokens/sec mean     12
Dolly Sequential sustained tokens/sec sd       12
Dolly CPU percent peak                         12
Dolly Sequential sustained samples/sec sd      12
Dolly CPU percent avg mean                     12
Dolly TTFT ms mean                             12
llama_bench n_prompt                           12
Dolly RAM used MB peak                         12
Dolly Process RAM MB peak                      12


In [68]:
if not all_samples_df.empty:
    print("Samples shape:", all_samples_df.shape)
    print("\nDatasets in samples:", sorted(all_samples_df["dataset"].dropna().unique()))
    print("\nRows by model/dataset:")
    display(
        all_samples_df.groupby(["model_dir", "dataset"])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values(["model_dir", "dataset"])
    )

    print("\nNull counts:")
    display(all_samples_df.isna().sum().sort_values(ascending=False).head(30))

Samples shape: (12300, 27)

Datasets in samples: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']

Rows by model/dataset:


,model_dir,dataset,n_rows
0,Llama-3.2-1B-Instruct-Q8_0,dolly,100
1,Llama-3.2-1B-Instruct-Q8_0,gsm8k,1000
2,Llama-3.2-1B-Instruct-Q8_0,hellaswag,1000
3,Llama-3.2-1B-Instruct-Q8_0,mmlu,1000
4,Phi-4-mini-instruct-Q4_K_M,dolly,100
5,Phi-4-mini-instruct-Q4_K_M,gsm8k,1000
6,Phi-4-mini-instruct-Q4_K_M,hellaswag,1000
7,Phi-4-mini-instruct-Q4_K_M,mmlu,1000
8,gemma-3-1b-it-Q8_0,dolly,100
9,gemma-3-1b-it-Q8_0,gsm8k,2000



Null counts:


context                  12207
instruction              12000
choices                   4300
parsed_prediction          372
question                   300
gold_answer                300
correct                    300
run_idx                      0
dataset_name                 0
sample_idx                   0
model_name                   0
prompt                       0
raw_output                   0
output_tokens                0
ttft_ms                      0
end_to_end_latency_ms        0
tokens_per_sec               0
cpu_percent_avg              0
cpu_percent_peak             0
ram_used_mb_peak             0
process_ram_mb_peak          0
temp_c_avg                   0
temp_c_peak                  0
model_dir                    0
run_id                       0
dataset                      0
source_path                  0
dtype: int64

In [69]:
if not all_samples_df.empty:
    numeric_checks = {}

    for col in [
        "ttft_ms",
        "end_to_end_latency_ms",
        "tokens_per_sec",
        "output_tokens",
        "cpu_percent_avg",
        "cpu_percent_peak",
        "ram_used_mb_peak",
        "process_ram_mb_peak",
        "temp_c_avg",
        "temp_c_peak",
    ]:
        if col in all_samples_df.columns:
            numeric_checks[col] = {
                "min": pd.to_numeric(all_samples_df[col], errors="coerce").min(),
                "max": pd.to_numeric(all_samples_df[col], errors="coerce").max(),
                "n_negative": (pd.to_numeric(all_samples_df[col], errors="coerce") < 0).sum(),
            }

    numeric_checks_df = pd.DataFrame(numeric_checks).T
    numeric_checks_df

In [70]:
if not all_samples_df.empty and "correct" in all_samples_df.columns:
    correctness_by_dataset = (
        all_samples_df.groupby("dataset")["correct"]
        .agg(["count", "nunique"])
        .reset_index()
    )
    correctness_by_dataset

In [71]:
if not all_samples_df.empty and "parsed_prediction" in all_samples_df.columns:
    display(
        all_samples_df.groupby("dataset")["parsed_prediction"]
        .apply(lambda s: s.notna().mean())
        .rename("parsed_prediction_non_null_rate")
        .reset_index()
    )

,dataset,parsed_prediction_non_null_rate
0,dolly,0.000
1,gsm8k,0.982
2,hellaswag,1.000
3,mmlu,1.000


In [72]:
if not all_samples_df.empty and {"model_dir", "run_id", "dataset", "run_idx"}.issubset(all_samples_df.columns):
    per_measured_run_counts = (
        all_samples_df.groupby(["model_dir", "run_id", "dataset", "run_idx"])
        .size()
        .rename("n_samples")
        .reset_index()
        .sort_values(["model_dir", "run_id", "dataset", "run_idx"])
    )
    per_measured_run_counts.head(50)

In [73]:
if not all_samples_df.empty and {"model_dir", "run_id", "dataset", "run_idx"}.issubset(all_samples_df.columns):
    count_consistency = (
        per_measured_run_counts.groupby(["model_dir", "run_id", "dataset"])["n_samples"]
        .agg(["min", "max", "nunique"])
        .reset_index()
        .sort_values(["model_dir", "run_id", "dataset"])
    )
    count_consistency

In [74]:
if not all_samples_df.empty and "run_idx" in all_samples_df.columns:
    display(
        all_samples_df.groupby("dataset")["run_idx"]
        .agg(["min", "max", "nunique"])
        .reset_index()
    )

,dataset,min,max,nunique
0,dolly,1,10,10
1,gsm8k,1,10,10
2,hellaswag,1,10,10
3,mmlu,1,10,10


In [75]:
dolly_summary = all_summaries_df[all_summaries_df["dataset"] == "dolly"].copy()
if not dolly_summary.empty:
    print("Dolly columns:")
    for c in dolly_summary.columns:
        print(c)

Dolly columns:
Model name
Dolly TTFT ms mean
Dolly TTFT ms sd
Dolly End-to-end latency ms mean
Dolly End-to-end latency ms sd
Dolly Tokens/sec mean
Dolly Tokens/sec sd
Dolly CPU percent avg mean
Dolly CPU percent avg sd
Dolly Temp C avg mean
Dolly Temp C avg sd
Dolly Sequential sustained tokens/sec mean
Dolly Sequential sustained tokens/sec sd
Dolly Sequential sustained samples/sec mean
Dolly Sequential sustained samples/sec sd
Dolly CPU percent peak
Dolly RAM used MB peak
Dolly Process RAM MB peak
Dolly Temp C peak
llama_bench model
llama_bench n_prompt
llama_bench n_gen
llama_bench prompt_tps
llama_bench gen_tps
llama_bench test_time_s
model_dir
run_id
dataset
source_path
Accuracy mean
Accuracy sd
TTFT ms mean
TTFT ms sd
End-to-end latency ms mean
End-to-end latency ms sd
Tokens/sec mean
Tokens/sec sd
CPU percent avg mean
CPU percent avg sd
Temp C avg mean
Temp C avg sd
Sequential sustained tokens/sec mean
Sequential sustained tokens/sec sd
Sequential sustained samples/sec mean
Seque

Observations to capture here:
- which models/runs are complete
- whether summary CSVs are consistently one-row files
- whether sample CSV counts are stable across measured runs
- whether schemas are consistent across runs
- whether Dolly summary needs special normalization
- any obvious null or metric anomalies